# RAG Demo: Retrieval-Augmented Generation

This notebook demonstrates the RAG (Retrieval-Augmented Generation) system used in the AI Agent Insure chatbot.

## What is RAG?

RAG combines:
1. **Retrieval**: Finding relevant information from a knowledge base using semantic search
2. **Augmentation**: Adding that context to the user's question
3. **Generation**: Using an LLM to generate an answer based on the retrieved context

## How it works:

1. User asks a question
2. System converts question to embeddings (vector representation)
3. System searches vector database for similar document chunks
4. System retrieves top-k most relevant chunks
5. System passes question + retrieved context to LLM
6. LLM generates answer based on the provided context

This approach reduces hallucinations by grounding answers in actual documents.


<hr />
# Prerequisites

Before running this notebook, ensure the following are set up:

## Required Setup

1. **Ollama must be running**
   ```bash
   ollama serve
   ```
   The notebook uses Ollama for both LLM inference and embeddings.

2. **Required Ollama models must be pulled**
   ```bash
   ollama pull llama3.2:3b
   ollama pull nomic-embed-text
   ```
   - `llama3.2:3b` - Used for LLM text generation
   - `nomic-embed-text` - Used for creating embeddings

3. **PDF file in place**
   - Ensure `presentation/knowledge-base/overwiew.pdf` exists
   - The notebook will create a vector store from this PDF on first run

<hr />

In [1]:
# Setup: Import required libraries
import warnings
warnings.filterwarnings('ignore')

%pip install -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
import glob

# Import LangChain components
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryMemory
from langchain.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import PDF processing
from pypdf import PdfReader

# Configuration
LLM_MODEL = "llama3.2:3b"
EMBEDDING_MODEL = "nomic-embed-text"
TEMPERATURE = 0.1 # Temp is a float between 0 and 1. 0 is deterministic, higher is more random.

# Used for splitting the documents into chunks
CHUNK_SIZE = 1800 # Number of characters in each chunk
CHUNK_OVERLAP = 350 # Number of characters to overlap between chunks

# Used for retrieving the top k chunks from the vector store
RETRIEVAL_K = 4

# Set up demo-specific paths (relative to current directory)
DEMO_PDF_DIR = Path.cwd() / "knowledge-base"
DEMO_CHROMA_DIR = Path.cwd() / "chroma_db_demo"

print("✅ All required libraries imported successfully")
print()
print("📋 Configuration:")
print(f"   📁 Demo PDF directory: {DEMO_PDF_DIR}")
print(f"   💾 Demo vector store: {DEMO_CHROMA_DIR}")
print(f"   🤖 LLM Model: {LLM_MODEL}")
print(f"   🔍 Embedding Model: {EMBEDDING_MODEL}")
print(f"   📊 Retrieval K: {RETRIEVAL_K}")
print(f"   🌡️  Temperature: {TEMPERATURE}")


✅ All required libraries imported successfully

📋 Configuration:
   📁 Demo PDF directory: /Users/rob/Development/CSCI_E-89_Deep_Learning_Fall_25/presentation/knowledge-base
   💾 Demo vector store: /Users/rob/Development/CSCI_E-89_Deep_Learning_Fall_25/presentation/chroma_db_demo
   🤖 LLM Model: llama3.2:3b
   🔍 Embedding Model: nomic-embed-text
   📊 Retrieval K: 4
   🌡️  Temperature: 0.1


## Step 1: Load PDF Documents

First, let's load and examine the PDF documents we'll be working with.


In [3]:
# Load PDF documents
print("📄 Loading PDF documents...")
print(f"   PDF directory: {DEMO_PDF_DIR}")
print()

# Load PDFs
pdf_files = glob.glob(f"{DEMO_PDF_DIR}/**/*.pdf", recursive=True)
print(f"   Found {len(pdf_files)} PDF file(s)")

documents = []
for pdf_path in pdf_files:
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    documents.append({
        'text': text,
        'source': os.path.basename(pdf_path)
    })

print(f"   ✅ Loaded {len(documents)} document(s)")
print()

# Store documents for display
loaded_documents = documents


📄 Loading PDF documents...
   PDF directory: /Users/rob/Development/CSCI_E-89_Deep_Learning_Fall_25/presentation/knowledge-base

   Found 1 PDF file(s)
   ✅ Loaded 1 document(s)



In [4]:
# Display loaded PDF content
print("=" * 70)
print("📄 LOADED PDF CONTENT")
print("=" * 70)
for i, doc in enumerate(loaded_documents, 1):
    print(f"\n{i}. Source: {doc['source']}")
    print(f"   Text length: {len(doc['text'])} characters")
    print(f"   Preview (first 500 chars):")
    print(f"   {doc['text'][:500]}...")
    print()

📄 LOADED PDF CONTENT

1. Source: overwiew.pdf
   Text length: 4010 characters
   Preview (first 500 chars):
   AI Agent Insure
Company Overview
Mission & Vision
AI Agent Insure is a next-generation insurance provider dedicated to safeguarding companies
developing and deploying agentic AI systems, autonomous robotics, advanced RAG
pipelines, and AI-native workflows. Our mission is to bridge the gap between technological
innovation and responsible risk management by offering the world’s most comprehensive
AI-native insurance solutions.
We envision a future where AI agents—autonomous, semi-autonomous, and m...



## Step 2: Split Documents into Chunks

Large documents need to be split into smaller chunks for efficient processing. This allows us to:
- Fit chunks into the LLM's context window
- Retrieve only relevant portions of documents
- Maintain semantic coherence within each chunk


In [5]:
# Split documents into chunks
print("📝 Splitting documents into chunks...")
print(f"   Chunk size: {CHUNK_SIZE} characters")
print(f"   Chunk overlap: {CHUNK_OVERLAP} characters")
print()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

all_chunks = []
all_metadatas = []
for doc in documents:
    chunks = text_splitter.split_text(doc['text'])
    all_chunks.extend(chunks)
    all_metadatas.extend([{'source': doc['source']} for _ in chunks])

print(f"   ✅ Created {len(all_chunks)} chunks from {len(documents)} document(s)")
print()

# Store chunks for display
created_chunks = all_chunks
created_metadatas = all_metadatas


📝 Splitting documents into chunks...
   Chunk size: 1800 characters
   Chunk overlap: 350 characters

   ✅ Created 3 chunks from 1 document(s)



### View Example Chunks

Here are some example chunks created from the documents:


In [6]:
# Display example chunks
print("=" * 70)
print(f"📝 TEXT CHUNKS ({len(created_chunks)} total)")
print("=" * 70)
print()
print(f"Chunk size: {CHUNK_SIZE} characters")
print(f"Chunk overlap: {CHUNK_OVERLAP} characters")
print()
print("Example chunks (first 3):")
for i, (chunk, metadata) in enumerate(zip(created_chunks[:3], created_metadatas[:3]), 1):
    print(f"\n{i}. Source: {metadata['source']}")
    print(f"   Length: {len(chunk)} characters")
    print(f"   Preview:")
    print(f"   {chunk[:300]}...")
    print()
print(f"\nAverage chunk size: {sum(len(c) for c in created_chunks) // len(created_chunks)} characters")


📝 TEXT CHUNKS (3 total)

Chunk size: 1800 characters
Chunk overlap: 350 characters

Example chunks (first 3):

1. Source: overwiew.pdf
   Length: 1787 characters
   Preview:
   AI Agent Insure
Company Overview
Mission & Vision
AI Agent Insure is a next-generation insurance provider dedicated to safeguarding companies
developing and deploying agentic AI systems, autonomous robotics, advanced RAG
pipelines, and AI-native workflows. Our mission is to bridge the gap between te...


2. Source: overwiew.pdf
   Length: 1726 characters
   Preview:
   technical risks of modern autonomous and agentic AI environments:
 Agentic AI Liability Insurance
 Autonomous System & Robotics Coverage
 AI Infrastructure & Ops Protection
 Model & Data Security Insurance
 Synthetic Data & Dataset Integrity Coverage
 Intellectual Property & Output Protection
...


3. Source: overwiew.pdf
   Length: 1083 characters
   Preview:
   adopt increasingly autonomous systems—ranging from AI copilots to AV fleets—the

## Step 3: Create Vector Store with Embeddings

Now we'll convert text chunks into vector embeddings and store them in a vector database. This enables semantic search - finding documents based on meaning, not just keywords.


In [7]:
# Create or load vector store
print("🔄 Creating embeddings and vector store...")
print()

# Check if vector store exists
if not os.path.exists(DEMO_CHROMA_DIR):
    print("   Creating new vector store...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    vectorstore = Chroma.from_texts(
        texts=all_chunks,
        embedding=embeddings,
        metadatas=all_metadatas,
        persist_directory=str(DEMO_CHROMA_DIR)
    )
    print("   ✅ Vector store created!")
else:
    print("   Loading existing vector store...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    vectorstore = Chroma(
        persist_directory=str(DEMO_CHROMA_DIR),
        embedding_function=embeddings
    )
    print("   ✅ Vector store loaded!")

print()
print(f"📊 Vector store contains {vectorstore._collection.count()} document chunks")


🔄 Creating embeddings and vector store...

   Creating new vector store...
   ✅ Vector store created!

📊 Vector store contains 3 document chunks


### View Vector Store Information

Information about the embeddings and vector store:


In [8]:
# Display vector store information
print("=" * 70)
print("🔢 VECTOR STORE INFORMATION")
print("=" * 70)
print()
print(f"📊 Total vectors: {vectorstore._collection.count()}")
print(f"🔍 Embedding model: {EMBEDDING_MODEL}")
print(f"📏 Embedding dimensions: ~768 (nomic-embed-text)")
print(f"💾 Persistence directory: {DEMO_CHROMA_DIR}")
print()
print("ℹ️  Each chunk is converted to a vector (embedding) that captures")
print("   the semantic meaning of the text. Similar texts have similar vectors.")


🔢 VECTOR STORE INFORMATION

📊 Total vectors: 3
🔍 Embedding model: nomic-embed-text
📏 Embedding dimensions: ~768 (nomic-embed-text)
💾 Persistence directory: /Users/rob/Development/CSCI_E-89_Deep_Learning_Fall_25/presentation/chroma_db_demo

ℹ️  Each chunk is converted to a vector (embedding) that captures
   the semantic meaning of the text. Similar texts have similar vectors.


## Step 4: Setup RAG Chain

Now we'll set up the RAG (Retrieval-Augmented Generation) chain that combines:
- **Retriever**: Finds relevant chunks from the vector store
- **LLM**: Generates answers based on retrieved context
- **Memory**: Maintains conversation history


In [9]:
# Setup RAG chain
print("🔧 Setting up RAG chain...")
print()

# Initialize LLM
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=TEMPERATURE,
    timeout=60.0
)

# Setup memory for conversation history
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
    max_token_limit=2000
)

# Custom prompt template
custom_template = """Answer based ONLY on the provided context. If the answer isn't in the context, say "I don't have that information in the available documents."

Context:
{context}

Question: {question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=custom_template,
    input_variables=["context", "chat_history", "question"]
)

# Setup retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVAL_K}
)

# Create RAG chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    verbose=False
)

print("✅ RAG chain ready!")
print(f"   Retrieving top {RETRIEVAL_K} chunks per query")
print(f"   Using LLM: {LLM_MODEL}")
print(f"   Temperature: {TEMPERATURE}")


🔧 Setting up RAG chain...

✅ RAG chain ready!
   Retrieving top 4 chunks per query
   Using LLM: llama3.2:3b
   Temperature: 0.1


/var/folders/xm/vwzlcq8n3n72g3j519crrygr0000gn/T/ipykernel_33829/2982959023.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(


## Step 5: Ask a Question

Let's ask a question about the company or products. The system will:
1. Convert the question to embeddings
2. Find the most relevant document chunks using semantic search
3. Pass the question and retrieved context to the LLM
4. Generate an answer based on the provided context


In [10]:
# Sample question (adjust based on your PDF content)
question = "What does this company do?"

print(f"❓ Question: {question}")
print()
print("🔄 Processing query through RAG pipeline...")
print()

# Execute RAG query
result = qa_chain({"question": question})

# Extract components
answer = result['answer']
source_documents = result['source_documents']

# Extract unique sources
sources = list(set([doc.metadata.get('source', 'Unknown') for doc in source_documents]))


❓ Question: What does this company do?

🔄 Processing query through RAG pipeline...



/var/folders/xm/vwzlcq8n3n72g3j519crrygr0000gn/T/ipykernel_33829/2862509164.py:10: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"question": question})


## Step 6: View Retrieved Chunks

Before generating the answer, let's see which chunks were retrieved from the vector store based on semantic similarity:


In [11]:
# Show retrieved chunks that were used as context
print("=" * 70)
print(f"🔍 RETRIEVED CHUNKS (used as context for LLM)")
print("=" * 70)
print()
print(f"These {len(source_documents)} chunks were retrieved based on semantic similarity to the question:")
print()

for i, doc in enumerate(source_documents, 1):
    source = doc.metadata.get('source', 'Unknown')
    chunk_text = doc.page_content
    print(f"{i}. Source: {source}")
    print(f"   Length: {len(chunk_text)} characters")
    print(f"   Content:")
    print(f"   {chunk_text[:500]}...")
    print()


🔍 RETRIEVED CHUNKS (used as context for LLM)

These 3 chunks were retrieved based on semantic similarity to the question:

1. Source: overwiew.pdf
   Length: 1787 characters
   Content:
   AI Agent Insure
Company Overview
Mission & Vision
AI Agent Insure is a next-generation insurance provider dedicated to safeguarding companies
developing and deploying agentic AI systems, autonomous robotics, advanced RAG
pipelines, and AI-native workflows. Our mission is to bridge the gap between technological
innovation and responsible risk management by offering the world’s most comprehensive
AI-native insurance solutions.
We envision a future where AI agents—autonomous, semi-autonomous, and m...

2. Source: overwiew.pdf
   Length: 1083 characters
   Content:
   adopt increasingly autonomous systems—ranging from AI copilots to AV fleets—the need for
specialized risk-transfer solutions grows. Traditional insurance models are not built for the
complexity or scale of these systems, creating a critical

## Step 7: Display Generated Answer

Now let's see the answer generated by the LLM using the retrieved context:


In [12]:
# Display the answer
print("=" * 70)
print("💬 GENERATED ANSWER")
print("=" * 70)
print(answer)
print()
print("=" * 70)
print(f"📚 SOURCES ({len(source_documents)} chunks from {len(sources)} document(s))")
print("=" * 70)
for i, source in enumerate(sources, 1):
    print(f"{i}. {source}")


💬 GENERATED ANSWER
AI Agent Insure is a next-generation insurance provider dedicated to safeguarding companies developing and deploying agentic AI systems, autonomous robotics, advanced RAG pipelines, and AI-native workflows.

📚 SOURCES (3 chunks from 1 document(s))
1. overwiew.pdf


## Step 8: Understanding the RAG Pipeline

Let's examine what happened behind the scenes in the RAG pipeline:


In [13]:
# Show the complete RAG pipeline breakdown
print("=" * 70)
print("🔍 RAG PIPELINE BREAKDOWN")
print("=" * 70)
print()
print("1. 📝 Question Processing:")
print(f"   Question: '{question}'")
print(f"   → Converted to embeddings using: {EMBEDDING_MODEL}")
print()
print("2. 🔎 Semantic Search:")
print(f"   → Searched {vectorstore._collection.count()} document chunks")
print(f"   → Retrieved top {len(source_documents)} most similar chunks")
print(f"   → Search type: similarity (cosine similarity)")
print()
print("3. 📚 Context Assembly:")
print(f"   → Combined {len(source_documents)} chunks into context")
print(f"   → Total context length: ~{sum(len(doc.page_content) for doc in source_documents)} characters")
print()
print("4. 🤖 LLM Generation:")
print(f"   → Model: {LLM_MODEL}")
print(f"   → Temperature: {TEMPERATURE} (lower = more focused)")
print(f"   → Prompt: Question + Retrieved Context")
print(f"   → Generated answer grounded in retrieved documents")
print()
print("5. ✅ Result:")
print(f"   → Answer generated from {len(sources)} source document(s)")
print(f"   → Answer length: {len(answer)} characters")


🔍 RAG PIPELINE BREAKDOWN

1. 📝 Question Processing:
   Question: 'What does this company do?'
   → Converted to embeddings using: nomic-embed-text

2. 🔎 Semantic Search:
   → Searched 3 document chunks
   → Retrieved top 3 most similar chunks
   → Search type: similarity (cosine similarity)

3. 📚 Context Assembly:
   → Combined 3 chunks into context
   → Total context length: ~4596 characters

4. 🤖 LLM Generation:
   → Model: llama3.2:3b
   → Temperature: 0.1 (lower = more focused)
   → Prompt: Question + Retrieved Context
   → Generated answer grounded in retrieved documents

5. ✅ Result:
   → Answer generated from 1 source document(s)
   → Answer length: 206 characters


## Step 9: Try Another Question

Let's ask a different question to see how the system retrieves different documents and generates a new answer:


In [14]:
# Try a different question (adjust based on your PDF content)
question2 = "What products does this company offer?"

print(f"❓ Question: {question2}")
print()
print("🔄 Processing...")
print()

result2 = qa_chain({"question": question2})
answer2 = result2['answer']
source_docs2 = result2['source_documents']
sources2 = list(set([doc.metadata.get('source', 'Unknown') for doc in source_docs2]))

print("=" * 70)
print("💬 ANSWER")
print("=" * 70)
print(answer2)
print()
print("=" * 70)
print(f"📚 SOURCES ({len(source_docs2)} chunks from {len(sources2)} document(s))")
print("=" * 70)
for i, source in enumerate(sources2, 1):
    print(f"{i}. {source}")


❓ Question: What products does this company offer?

🔄 Processing...

💬 ANSWER
AI Agent Insure offers a comprehensive portfolio of insurance products tailored to the unique technical risks of modern autonomous and agentic AI environments, including:

1. Agentic AI Liability Insurance
2. Autonomous System & Robotics Coverage
3. AI Infrastructure & Ops Protection
4. Model & Data Security Insurance
5. Synthetic Data & Dataset Integrity Coverage
6. Intellectual Property & Output Protection
7. Compliance & Regulatory Shield
8. Incident Response & AI Crisis Management
9. Agentic Workflow Uptime Insurance

📚 SOURCES (3 chunks from 1 document(s))
1. overwiew.pdf


In [15]:
# Cleanup
!rm -rf chroma_db_demo/